# 03 — Базовые модели: NDSI и Random Forest

Контрольный эксперимент: сравнение с базовыми моделями доказывает
научную ценность U-Net (Месяц 4 плана).


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

from src import config, metrics

X_train = np.load(config.DATA_PATCHES / 'X_train.npy')
X_test  = np.load(config.DATA_PATCHES / 'X_test.npy')
y_train = np.load(config.DATA_PATCHES / 'y_train.npy')
y_test  = np.load(config.DATA_PATCHES / 'y_test.npy')

print('N каналов:', X_train.shape[-1], '(ожидается', config.N_CHANNELS, ')')


In [ ]:
def flatten_for_ml(X, y):
    N, H, W, C = X.shape
    return X.reshape(N * H * W, C), y.reshape(N * H * W)

X_train_flat, y_train_flat = flatten_for_ml(X_train, y_train)
X_test_flat,  y_test_flat  = flatten_for_ml(X_test, y_test)

print(f'Обучающих пикселей: {X_train_flat.shape[0]:,}')


## Модель 1: NDSI пороговая классификация

Индекс NDSI берётся по `config.BAND_INDEX['NDSI']` — это устраняет
ошибку из черновика плана с "магическим" индексом 7.


In [ ]:
NDSI_INDEX = config.BAND_INDEX['NDSI']
results_rows = []

for threshold in config.NDSI_THRESHOLDS:
    ndsi_vals = X_test_flat[:, NDSI_INDEX]
    y_pred_ndsi = (ndsi_vals > threshold).astype(int)
    m = metrics.evaluate_segmentation(y_test_flat, y_pred_ndsi)
    print(f"NDSI > {threshold}: F1={m['f1']:.3f}, Precision={m['precision']:.3f}, "
          f"Recall={m['recall']:.3f}, IoU={m['iou']:.3f}")
    results_rows.append({'threshold': threshold, **m})

best_row = max(results_rows, key=lambda r: r['f1'])
BEST_NDSI_THRESHOLD = best_row['threshold']
print(f'\nЛучший порог NDSI: {BEST_NDSI_THRESHOLD} (F1={best_row["f1"]:.3f})')

y_pred_ndsi_best = (X_test_flat[:, NDSI_INDEX] > BEST_NDSI_THRESHOLD).astype(int)
ndsi_metrics = metrics.evaluate_segmentation(y_test_flat, y_pred_ndsi_best)


## Модель 2: Случайный лес

In [ ]:
np.random.seed(config.RANDOM_SEED)
n_sample = min(1_000_000, len(X_train_flat))
idx = np.random.choice(len(X_train_flat), n_sample, replace=False)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=config.RANDOM_SEED,
)

print('Обучение случайного леса...')
rf.fit(X_train_flat[idx], y_train_flat[idx])
print('Случайный лес обучен!')

y_pred_rf = rf.predict(X_test_flat)
rf_metrics = metrics.evaluate_segmentation(y_test_flat, y_pred_rf)
print(f"Случайный лес: F1={rf_metrics['f1']:.3f}, Precision={rf_metrics['precision']:.3f}, "
      f"Recall={rf_metrics['recall']:.3f}, IoU={rf_metrics['iou']:.3f}")

feature_importance = sorted(zip(config.ALL_BAND_NAMES, rf.feature_importances_), key=lambda x: -x[1])
for name, imp in feature_importance:
    print(f'  {name}: {imp:.3f}')

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(rf, config.MODELS_DIR / 'random_forest.pkl')


## Сводная таблица (заполнится U-Net в ноутбуке 04)

In [ ]:
results = {
    'Метод': ['NDSI (порог)', 'Случайный лес'],
    'F1-score':  [ndsi_metrics['f1'], rf_metrics['f1']],
    'Precision': [ndsi_metrics['precision'], rf_metrics['precision']],
    'Recall':    [ndsi_metrics['recall'], rf_metrics['recall']],
    'IoU':       [ndsi_metrics['iou'], rf_metrics['iou']],
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
df_results.to_csv(config.TABLES_DIR / 'model_comparison.csv', index=False)
